# 10 — The test years, scored once

Everything about the model was decided on 2014–2024: the 40 inputs, the 8-fight window,
shrinkage, the Elo settings, the model settings. This notebook opens the test years,
**2025 onwards**, for the first and only time.

The recipe is frozen (`reviews/final_model_2026-10-17.md`): the model is refitted on
2014–2024 with exactly those choices (alphas and calibration refitted on the same years, as
the recipe says), then scored on the test fights once. **Nothing about the model changes
because of what this notebook shows.**

Reported as agreed: men's fights first, then all fights, each split into all fights / no
debutant / with a debutant. The betting market on the test years comes later, once its odds
can be matched; the predictions are saved here so that comparison uses exactly these numbers.

In [ ]:
import json, subprocess
import numpy as np
import pandas as pd

from ufc_predictor.config import RAW_DIR, TEST_START
from ufc_predictor.data.io import load_fighters, load_processed
from ufc_predictor.features.elo import load_elo_params, load_elo_weights
from ufc_predictor.modeling.evaluate import (breakdown, calibration_table, fit_baseline, has_debutant,
                                             is_womens_fight, predict_baseline)
from ufc_predictor.modeling.final import fit_final, predict, training_rows
from ufc_predictor.modeling.split import chronological_split
from ufc_predictor.modeling.tune import load_params

# the record: which code and which data this one look used
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain", "src", "reviews/final_model_2026-10-17.md"],
                       capture_output=True, text=True).stdout.strip()
assert not dirty, f"the model code has uncommitted changes - commit the frozen recipe first:\n{dirty}"
source = json.loads((RAW_DIR / "source" / "SOURCE.json").read_text())
params, elo_k, elo_weights = load_params(), load_elo_params(), load_elo_weights()
print(f"code {commit} | data {source['dataset']} v{source['version']} (to {source['last_event']})")
print(f"Elo K {elo_k}, weights {elo_weights}\nsettings {params}")

## 1. Fit on 2014–2024

In [ ]:
fights, fighters = load_processed("fights"), load_fighters()
bundle = fit_final(fights, fighters, until=TEST_START, params=params, elo_k=elo_k, elo_weights=elo_weights)
featured = bundle["featured"]
print(f"trained on {bundle['training_fights']:,} fights before {bundle['until']}; calibration stretch {bundle['stretch']:.3f}")

## 2. The one look

In [ ]:
_, _, test = chronological_split(featured)
train = training_rows(featured, TEST_START)
y = test["fighter_a_won"].astype(bool).to_numpy()
mens = ~is_womens_fight(test, history=featured).to_numpy()
debutant = has_debutant(test).to_numpy()

predictions = {"coin flip": np.full(len(test), 0.5)}
for column, name in [("diff_prior_win_rate", "win rate only"), ("diff_static_age_gap", "age only")]:
    baseline = fit_baseline(train, train["fighter_a_won"].astype(bool), column)   # fitted on 2014-2024 only
    predictions[name] = predict_baseline(baseline, test, column)
predictions["our model"] = predict(bundle, test)
print(f"test fights: {len(test):,} ({mens.sum():,} men's, {debutant.sum():,} with a debutant), "
      f"{test['event_date'].min().date()} to {test['event_date'].max().date()}")
print("\nMEN'S FIGHTS")
mens_card = breakdown(predictions, y, debutant, keep=mens)
mens_card

In [ ]:
print("ALL FIGHTS")
all_card = breakdown(predictions, y, debutant)
all_card

In [ ]:
# are the percentages honest on fights the model never saw? (men's)
calibration_table(y[mens], predictions["our model"][mens], bins=5)

## 3. Keep the record

In [ ]:
out = test[["fights_url", "event_date", "fighter_a_name", "fighter_b_name", "fighter_a_won"]].copy()
out["p_fighter_a"] = predictions["our model"].round(6)
out["mens"], out["has_debutant"] = mens, debutant
out.to_csv("reviews/test_predictions_2026-10-17.csv", index=False)
record = {"code": commit, "data_version": source["version"], "data_last_event": source["last_event"],
          "trained_until": bundle["until"], "training_fights": bundle["training_fights"], "stretch": bundle["stretch"],
          "elo": {"k": elo_k, "weights": elo_weights}, "params": params,
          "mens": mens_card.reset_index().to_dict("records"), "all": all_card.reset_index().to_dict("records")}
json.dump(record, open("reviews/test_result_2026-10-17.json", "w"), indent=1, default=str)
print(f"saved {len(out):,} predictions and the scores")